In [1]:
# change to the Fgpt directory

%cd ..

/home/kardaneh/Fgpt


/home/kardaneh/fparser_env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from processor import Processor
from extractor import Extractor
from isolator import Isolator
from fparser.two import Fortran2003 as F23
from fparser.two.utils import walk
import logging
from typing import Generator, Optional, Type, Union,Dict,List,Literal
import numpy as np
import os

processor = Processor() 

Setting the path to the main code, target module, creating an instance of the Isolator, Extractor class

In [4]:
rest_of_path = "modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/"
target_module = "hydrol"
work = os.getenv("works")
isolator = Isolator(rest_of_path, target_module, work)
cls = Extractor(isolator.module_dir_sp, isolator.module_tree_sp)
cls.find_subroutines()
cls.extract_loop_indices()

INFO:root:Successfully parsed file: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol_org.f90
INFO:root:Successfully parsed string!


In [5]:
import ast
empty_ast = ast.Module(body=[], type_ignores=[])
# Let's say that the AST fortran and AST python works in a similar manner, this will allows us to do the following:
# - modify/add to the python AST structure directly without the need to work with indentation, thus during the translation we modify the AST
#   strucutre
# - useful when integrating with multiples files thus allowing us to modify between both files

# Let's start by creating a python AST structure for the code template
# Then compare these structures, to do so we will use the code template from the module_global file which is set as out_module
subroutine_key = "hydrol_diag_soil" # hydrol_diag_soil"
out_module = processor.out_module_fortran(subroutine_key) # Create a template Fortran module
print(out_module)

INFO:root:Successfully parsed module code



MODULE module_global
  IMPLICIT NONE
  INTEGER, PARAMETER :: i_std = 4
  INTEGER, PARAMETER :: r_std = 8
  INTEGER(KIND = i_std), PARAMETER :: nsnow = 3
  INTEGER(KIND = i_std), PARAMETER :: nslm = 11
  INTEGER(KIND = i_std), PARAMETER :: nvm = 15
  INTEGER(KIND = i_std), PARAMETER :: nstm = 3
  INTEGER(KIND = i_std), PARAMETER :: kjpindex = 4717
  INTEGER :: ier
  INTEGER(KIND = i_std) :: ic0, ic
  REAL(KIND = r_std) :: icr, start_time, stop_time
  CONTAINS
  SUBROUTINE declaration_initialization
    OPEN(UNIT = 1363, FILE = '/home/kardaneh/Fgpt/benchmark/hydrol_diag_soil/global.bin', FORM = 'unformatted', STATUS = 'old')
    WRITE(*, *) '--- add the declaration and initialization in module global ---'
  END SUBROUTINE declaration_initialization
END MODULE module_global



In [22]:
def python_parser(code:str):
    try:
        tree = ast.parse(code)
        print("Code valid")
        return tree
    except SyntaxError as e:
        print(f'Syntax error: {e}')
        return None

In [23]:
module_global_templates = f"""
import numpy as np
from scipy.io import FortranFile
import os

i_std = np.int32   
r_std = np.float64

nsnow = i_std(3)
nslm = i_std(11)
nvm = i_std(15)
nstm = i_std(3)
kjpindex = i_std(4717)

ier = i_std(0)
ic0 = i_std(0)
ic = i_std(0)
icr = r_std(0.0)
start_time = r_std(0.0)
stop_time = r_std(0.0)

def declaration_initialization():
    print("--- add the declaration and initialization in module global ---")
"""
# THis is for the module_global python template 

In [24]:
global_python_template = python_parser(module_global_templates)

Code valid


In [25]:
print(ast.dump(global_python_template,indent=4))

Module(
    body=[
        Import(
            names=[
                alias(name='numpy', asname='np')]),
        ImportFrom(
            module='scipy.io',
            names=[
                alias(name='FortranFile')],
            level=0),
        Import(
            names=[
                alias(name='os')]),
        Assign(
            targets=[
                Name(id='i_std', ctx=Store())],
            value=Attribute(
                value=Name(id='np', ctx=Load()),
                attr='int32',
                ctx=Load())),
        Assign(
            targets=[
                Name(id='r_std', ctx=Store())],
            value=Attribute(
                value=Name(id='np', ctx=Load()),
                attr='float64',
                ctx=Load())),
        Assign(
            targets=[
                Name(id='nsnow', ctx=Store())],
            value=Call(
                func=Name(id='i_std', ctx=Load()),
                args=[
                    Constant(value=3)],
          

In [26]:
out_module.children[1]

Module(Module_Stmt('MODULE', Name('module_global')), Specification_Part(Implicit_Part(Implicit_Stmt('NONE')), Type_Declaration_Stmt(Intrinsic_Type_Spec('INTEGER', None), Attr_Spec_List(',', (Attr_Spec('PARAMETER'),)), Entity_Decl_List(',', (Entity_Decl(Name('i_std'), None, None, Initialization('=', Int_Literal_Constant('4', None))),))), Type_Declaration_Stmt(Intrinsic_Type_Spec('INTEGER', None), Attr_Spec_List(',', (Attr_Spec('PARAMETER'),)), Entity_Decl_List(',', (Entity_Decl(Name('r_std'), None, None, Initialization('=', Int_Literal_Constant('8', None))),))), Type_Declaration_Stmt(Intrinsic_Type_Spec('INTEGER', Kind_Selector('(', Name('i_std'), ')')), Attr_Spec_List(',', (Attr_Spec('PARAMETER'),)), Entity_Decl_List(',', (Entity_Decl(Name('nsnow'), None, None, Initialization('=', Int_Literal_Constant('3', None))),))), Type_Declaration_Stmt(Intrinsic_Type_Spec('INTEGER', Kind_Selector('(', Name('i_std'), ')')), Attr_Spec_List(',', (Attr_Spec('PARAMETER'),)), Entity_Decl_List(',', (Enti

In [27]:
subroutine_tree = cls.subroutines[subroutine_key] # The Fortran subroutine in the AST format

In [28]:
cls.call_within_sub # The procedure dependencies

defaultdict(set,
            {'hydrol_main': {'explicitsnow_main',
              'histwrite_p',
              'hydrol_alma',
              'hydrol_canop',
              'hydrol_flood',
              'hydrol_hydraulic_arch_tuzet_calc',
              'hydrol_nudge_mc_diag',
              'hydrol_nudge_snow',
              'hydrol_soil',
              'hydrol_vegupd'},
             'hydrol_vegupd': {'hydrol_tmc_update'},
             'hydrol_soil': {'hydrol_diag_soil',
              'hydrol_diag_soil_flux',
              'hydrol_nudge_mc',
              'hydrol_root_profile',
              'hydrol_soil_coef',
              'hydrol_soil_froz',
              'hydrol_soil_infilt',
              'hydrol_soil_setup',
              'hydrol_soil_smooth_over_mcs2',
              'hydrol_soil_smooth_under_mcr',
              'hydrol_soil_tridiag',
              'hydrol_split_soil'},
             'hydrol_nudge_snow': {'flinget',
              'flininfo',
              'scatter',
              'xios

In [29]:
cls.extract_intent(subroutine_key, subroutine_tree) # Determining the intent of the dummy variables in the procedure (IN, INOUT, OUT)

In [30]:
cls.clean_subroutine(subroutine_key, subroutine_tree) # Correct any incorrect intents, and warn if any intents are not used

In [31]:
cls.find_variables(subroutine_tree, subroutine_key) # Find variables in a procedure: local, global, or dummy.

In [32]:
cls.extract_names(subroutine_key)

Find the global variables — those that are neither dummy arguments nor local variables. The Navigator class searches in the current Fortran module, and if not found, it looks in the dependent modules to locate the variables. If a variable is allocatable, it also locates its allocation.

In [33]:
cls.find_global_variables(isolator.module_dir_sp, isolator.module_tree_sp, cls.var_global[subroutine_key], subroutine_key)

Searching for variable: tmc_litt_wet_mea ... ⏳
<tmc_litt_wet_mea> is found in <<hydrol>> of the module <<< hydrol >>>
REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:) :: tmc_litt_wet_mea
<tmc_litt_wet_mea> is found in <<hydrol_init>> of the module <<< hydrol >>>
ALLOCATE(tmc_litt_wet_mea(kjpindex), STAT = ier)
The containing directory is: /scratchu/kardaneh/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/
✅ Variable found!

Searching for variable: un ... ⏳
Add! Module ioipsl is added into the queue.
Add! Module xios_orchidee is added into the queue.
Add! Module constantes is added into the queue.
Add! Module time is added into the queue.
Add! Module constantes_soil is added into the queue.
Add! Module pft_parameters is added into the queue.
Add! Module sechiba_io_p is added into the queue.
Add! Module grid is added into the queue.
Add! Module explicitsnow is added into the queue.
Checking the child module .... ioipsl
Checking the child module .... xios_orchidee
Add! Module xios is ad

In [34]:
print(cls.dec_global[subroutine_key].keys())

dict_keys(['tmc_litt_wet_mea', 'un', 'vegtot', 'mcl', 'humrelv', 'tmc', 'frac_bare_ns', 'tmc_litt_dry_mea', 'soilmoist_liquid', 'imin', 'soil_wet_litter', 'mask_soiltile', 'vegtot_old', 'tmc_litter_res', 'tmc_litt_mea', 'zero', 'tmc_litter_awet', 'soilmoist_s', 'vegstressv', 'iice', 'imax', 'k_lin', 'tmc_litter', 'dr_ns', 'tmc_litter_sat', 'ok_freeze_cwrr', 'min_sechiba', 'profil_froz_hydro_ns', 'huit', 'nnobio', 'profil_froz_hydro', 'subsinksoil', 'soilmoist', 'dz', 'tmc_litter_adry', 'ae_ns', 'humtot', 'ru_ns', 'dh', 'mc', 'trois', 'soil_wet_ns'])


In [35]:
cls.extract_array_info(cls.dec_global[subroutine_key], cls.var_dummy[subroutine_key],subroutine_key)

INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: tmc_litt_wet_mea
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: mcl
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: humrelv
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: tmc
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: frac_bare_ns
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: tmc_litt_dry_mea
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm) :: soilmoist_liquid
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: soil_wet_litter
INFO:root:Combined statement: INTEGER(KIND = i_std), DIMENSION(kjpindex, nstm) :: mask_soiltile
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot_old
INFO:root:Com

In [36]:
# This allows the acqusition of all the variables in the order in which they should be read inside the attributes: read_declaration_in_routine,
# reads_in_read_routine
isolator.processor_sp.add_declarations(cls.dec_global[subroutine_key], cls.var_modif_info[subroutine_key])

INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: ae_ns
INFO:root:Successfully parsed statement: if(.not. allocated(ae_ns))then
ALLOCATE(ae_ns(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully parsed statement: if(.not. allocated(ae_ns_cpu))then
ALLOCATE(ae_ns_cpu(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully generated allocation statements
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(nslm) :: dh
INFO:root:Successfully parsed statement: if(.not. allocated(dh))then
ALLOCATE(dh(nslm), STAT = ier)
end if
INFO:root:Successfully generated allocation statements
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: dr_ns
INFO:root:Successfully parsed statement: if(.not. allocated(dr_ns))then
ALLOCATE(dr_ns(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully parsed statement: if(.not. allocated(dr_ns_cpu))then
ALLOCATE(dr_ns_cpu(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully generated 

**Till here we primarily do what has been done on the isolator class, after this is usually done in the update_module_global method but instead we transform from FORTRAN to PYTHON**

In [37]:
# Now to retrieve the variable in order. 
def retreive_variable_order(read_declarations):
    variable_order = []
    for read_dec in read_declarations:
        read_stmt = walk(read_dec,F23.Input_Item_List)
        for item in read_stmt:
            variable_order.append(item.children[0].string)
    # print(variable_order)

    return variable_order

variable_order = retreive_variable_order([isolator.processor_sp.reads_in_decleration_routine,isolator.processor_sp.reads_in_read_routine])
print(f'List of variables to be initialized: {variable_order}')

List of variables to be initialized: ['imax', 'ok_freeze_cwrr', 'ae_ns', 'dh', 'dr_ns', 'dz', 'frac_bare_ns', 'humrelv', 'humtot', 'k_lin', 'mask_soiltile', 'mc', 'mcl', 'profil_froz_hydro', 'profil_froz_hydro_ns', 'ru_ns', 'soil_wet_litter', 'soil_wet_ns', 'soilmoist', 'soilmoist_liquid', 'soilmoist_s', 'subsinksoil', 'tmc', 'tmc_litt_dry_mea', 'tmc_litt_mea', 'tmc_litt_wet_mea', 'tmc_litter', 'tmc_litter_adry', 'tmc_litter_awet', 'tmc_litter_res', 'tmc_litter_sat', 'vegstressv', 'vegtot', 'vegtot_old']


In [38]:
import ast

# We can remove this and only keep the ast_nodes and then once we have added the nodes onto the primary node(code template) we can iterate through them to 
# fix the missing lineno, col_offset using the ast.fix_missing_locations
def set_missing_locations(node, lineno=1, col_offset=0):
    if not hasattr(node, 'lineno'):
        node.lineno = lineno
    if not hasattr(node, 'col_offset'):
        node.col_offset = col_offset

    for n in ast.iter_child_nodes(node):
        set_missing_locations(n, lineno, col_offset)  # Recurse and set for all children

    return node

# Now we need to add the lineno and the rest of three elements
def convert_SPECIFICATION_PART(dec_global:Dict,fix_loc:bool=False,cls_mode:bool=False):
    ast_nodes = []

    kind_map = {
        'r_std': 'np.float64',  
        'i_std': 'np.int32'
    }

    subroutine_name = 'hydrol_diag_soil'
    target = None
    # Verify is the dec_global is an instance of Dict of Dict 
    assert isinstance(dec_global[subroutine_name], Dict), "dec_global should be a Dict of Dict"

    for var_name, declarations in dec_global[subroutine_name].items():
        # First is that we verify if we have a type declaration stmt and it's allocation stmt -> requires combine_allocate_declarations
        # in order to retrieve a combined and a proper allocation table variable

        # ANOTHER possibility with the first condition is that if we have intent within the declarations and also a length of 2 the
        # combine_allocate_declaration method will just remove it since and return a new formatted variable( look at the return value
        # of the method)
        if len(declarations) == 2:
            declarations = Processor().combine_allocate_declaration(declarations)
            # print(declarations)
        else: 
            # Need to verify if the one of the declarations has an INTENT/SAVE/PUBLIC among them thus we need to remove it before
            # transformation.
                
            declarations = Processor().remove_intent_and_save(declarations)
            # print(declarations[0])
        for nodes in walk(declarations, F23.Type_Declaration_Stmt):
            value = None
            attr_spec = [param.string for param in walk(nodes,F23.Attr_Spec)] # This looks for the Attr specification such as ALLOCATBLE or PARAMETER
            allocation_spec = [param.children[0] for param in walk(nodes,F23.Dimension_Attr_Spec) if param.children[0] == 'DIMENSION']
            # print(allocation_spec)
            kind_selec = [param.string for param in walk(nodes, F23.Kind_Selector)]

            # print(kind_selec)
            intrinsic_type_spec,_,entity_decl_list = nodes.children # This gives out a tuple
            entity_decls = entity_decl_list.children
            for entity_decl in entity_decls:
                var_name, _,_, initialization = entity_decl.children

                if cls_mode:
                    target = ast.Attribute(
                            value = ast.Name(id="self",ctx=ast.Load()),
                            attr = var_name.string, ctx = ast.Store()
                    )
                else:
                    target = ast.Name(id=var_name.string, ctx=ast.Store())
                
                if initialization is not None:
                    _,value = initialization.children
    
                if len(kind_selec)!= 0:
                    kind_selection = walk(nodes,F23.Kind_Selector)[0].children[1]
        
                if 'PARAMETER' in attr_spec:
                    if intrinsic_type_spec.children[0] == 'INTEGER' and len(kind_selec)==0: 
                        # These are only for element thats has parameters has
                        # has no kind inside need to change this assimilate the numpy format
                        if value is not None:
                            assign = ast.Assign(
                                        targets=[target],
                                        value=ast.Constant(value=int(value.children[0].strip()))
                                    )
                        else:
                            assign = ast.Assign(
                                        targets=[target],
                                        value=ast.Constant(value=0)
                                    )
                        ast_nodes.append(assign)
                        # print(f'Dtype:{intrinsic_type_spec}, value:{entity_decl_list}')
                    elif len(kind_selec) != 0: # IF the keyword KIND is present  
                        
                        np_dtype = kind_map.get(kind_selection.string, 'np.float64')
                        # Create: var = np.array(value, dtype=np_dtype)
                        idx,attr = np_dtype.split('.')
                        value_ = None
                        val = None
                        if value is not None:
                                
                            if len(value.children) > 2:
                                num1,_, num2 = value.children
                                value_ = ast.BinOp(
                                    left=ast.Constant(value=float(num1.string)),
                                    op=ast.Mult(),
                                    right=ast.Constant(value=float(num2.string))
                                )
                            else:
                                if attr == "int32":
                                    value_ = ast.Constant(value=int(value.children[0]))
                                else:
                                    value_ = ast.Constant(value=float(value.children[0]))

                            val = ast.Call(
                                func=ast.Attribute(value=ast.Name(id=idx, ctx=ast.Load()), attr=attr, ctx=ast.Load()),
                                args=[value_],
                                keywords = []
                                    # keywords=[
                                    #     ast.keyword(arg='dtype', value=ast.Attribute(value=ast.Name(id=idx, ctx=ast.Load()), attr=attr, ctx=ast.Load()))
                                    # ]
                            )
                        else:
                            val = ast.Call(func=ast.Attribute(value=ast.Name(id=idx, ctx=ast.Load()), attr=attr, ctx=ast.Load()),
                                            args=[ast.Constant(value=0)],
                                            keywords = []
                                        )
                            
                        assign = ast.Assign(
                                targets=[target],
                                value=val
                            )
                
                    ast_nodes.append(assign)
                        
                elif 'DIMENSION' in allocation_spec:
                    dimension_spec = None
                    dimensions_spec_list = walk(walk(nodes,F23.Dimension_Attr_Spec),F23.Explicit_Shape_Spec_List)
                    shape = []
                    left,right = None,None
                    constant_right = None
                    arg_shape = None
                    # print(dimensions_spec_list[0].children)
                    for child in dimensions_spec_list[0].children:
                        limits = child.tostr().split(':')
                        lb = limits[0]
                        if len(limits) > 1:
                            ub = limits[1]
                            if lb:
                                constant_right = ast.Constant(1)
                            
                            if cls_mode:
                                left = ast.Attribute( # Upper bound 
                                    value = ast.Name(id = 'self', ctx = ast.Load()),
                                    attr = ub,
                                    ctx = ast.Load())
                                right = ast.Attribute( # Upper bound 
                                    value = ast.Name(id = 'self', ctx = ast.Load()),
                                    attr = lb,
                                    ctx = ast.Load())
                            else:
                                left = ast.Name(id = ub, ctx = ast.Load())
                                right = ast.Name(id = lb, ctx = ast.Load())
                           
                            arg_shape = ast.BinOp(
                                left = ast.BinOp(
                                    left = left,
                                    op = ast.Sub(),
                                    right = right),
                                op = ast.Add(),
                                right = constant_right)
                            # print(ast.dump(arg_shape,indent=4))
                            shape.append(arg_shape)
                        else:
                            if cls_mode:
                                arg_shape = ast.Attribute(
                                    value = ast.Name(id = 'self',ctx = ast.Load()),
                                    attr = f"{lb}",
                                    ctx = ast.Load()
                                )
                            else:
                                arg_shape = ast.Name(f"{lb}")
                            shape.append(arg_shape)
                        # dimensions = ",".join([sh for sh in shape])

                    np_dtype = kind_map.get(kind_selection.string, 'np.float64')
                    # Create: var = np.array(value, dtype=np_dtype)
                    idx,attr = np_dtype.split('.')
                    np_call = ast.Call(
                        func=ast.Attribute(value=ast.Name(id='np', ctx=ast.Load()), attr='empty', ctx=ast.Load()),
                        args=[ast.Tuple(elts=shape, ctx=ast.Load())],
                        keywords=[ ast.keyword(arg='dtype', value = ast.Attribute(value=ast.Name(id=idx, ctx=ast.Load()), attr=attr, ctx=ast.Load()))]
                    )
                        
                    assign = ast.Assign(
                        targets=[target],
                        value=np_call
                    )
                    ast_nodes.append(assign)
                
                else: # Cases where the variable is not a PARAMETER nor ALLOCATABLE present
                    if intrinsic_type_spec.children[0] == 'LOGICAL' and len(kind_selec)==0: # SET AS AN EXCEPTIONAL CASE
                        bool_call = ast.Call(
                                        func=ast.Attribute(value=ast.Name(id='np', ctx=ast.Load()), attr='bool', ctx=ast.Load()),
                                        args=[ast.Constant(value=False)],
                                        keywords=[]
                                    )
                        assign = ast.Assign(
                                        targets=[target],
                                        value=bool_call
                                    )
                        ast_nodes.append(assign)
                    
                    elif len(kind_selec)==0:
                        idx,attr = None,None
                        if value is None: # Example cases : INTEGER :: ier
                            if intrinsic_type_spec.children[0] == "INTEGER":
                                np_dtype = kind_map.get('i_std', 'np.float64')
                                # Create: var = np.type(value) where type can be either integer or float 
                                idx,attr = np_dtype.split('.') 
                                    
                            elif intrinsic_type_spec.children[0] == "REAL":
                                np_dtype = kind_map.get('r_std', 'np.float64')
                                idx,attr = np_dtype.split('.')
                                
                            assign = ast.Assign(
                                        targets=[target],
                                        value=ast.Call( func=ast.Attribute(value=ast.Name(id=idx, ctx=ast.Load()), attr=attr, ctx=ast.Load()),
                                                        args=[ast.Constant(value=0)],
                                                        keywords = []
                                                      )
                                        )
                            ast_nodes.append(assign)
        if fix_loc:
            for node in ast_nodes:
                set_missing_locations(node)
            
    return ast_nodes
            
# ast_nodes = convert_SPECIFICATION_PART(tree)

ast_nodes = convert_SPECIFICATION_PART(cls.dec_global,fix_loc=True,cls_mode=True)
print(f'Size fortran global variables : {len(cls.var_global[subroutine_key])}, Size python global variables: {len(ast_nodes)}')
for nodes in ast_nodes:
     module = ast.Module(body=[nodes], type_ignores=[])
    # print(f'\n Python AST: {ast.dump(module,indent=4)}')
     print(f'\n Python code: {ast.unparse(module)}')


INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: tmc_litt_wet_mea
INFO:root:Successfully removed INTENT and SAVE attributes from statements
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: mcl
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: humrelv
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: tmc
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: frac_bare_ns
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: tmc_litt_dry_mea
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm) :: soilmoist_liquid
INFO:root:Successfully removed INTENT and SAVE attributes from statements
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: soil_wet_litter
INFO:root:Combined statement: INTEGER(KIND = 

Size fortran global variables : 42, Size python global variables: 42

 Python code: self.tmc_litt_wet_mea = np.empty((self.kjpindex,), dtype=np.float64)

 Python code: self.un = np.float64(1.0)

 Python code: self.vegtot = np.empty((self.kjpindex,), dtype=np.float64)

 Python code: self.mcl = np.empty((self.kjpindex, self.nslm, self.nstm), dtype=np.float64)

 Python code: self.humrelv = np.empty((self.kjpindex, self.nvm, self.nstm), dtype=np.float64)

 Python code: self.tmc = np.empty((self.kjpindex, self.nstm), dtype=np.float64)

 Python code: self.frac_bare_ns = np.empty((self.kjpindex, self.nstm), dtype=np.float64)

 Python code: self.tmc_litt_dry_mea = np.empty((self.kjpindex,), dtype=np.float64)

 Python code: self.soilmoist_liquid = np.empty((self.kjpindex, self.nslm), dtype=np.float64)

 Python code: self.imin = np.int32(1)

 Python code: self.soil_wet_litter = np.empty((self.kjpindex, self.nstm), dtype=np.float64)

 Python code: self.mask_soiltile = np.empty((self.kjpindex, sel

In [39]:
# Test out the AST python with the python global_python_template

nodes = ast.walk(global_python_template) # It's a generator of object that requires to go through them
func_def = {}
for node in ast.iter_child_nodes(global_python_template):
    if isinstance(node, ast.Import):
        names = node.names[0] # This gives a list of alias object each containing either asname or name
        # print(names.name)
    if isinstance(node,ast.Assign): # This has two items: targets and values
        target = [name.id for name in node.targets] # This returns a List of tuple which has the id(name) and the ctx which either to store or load
        # print(targets[0].id)
        # The values can either be an attribute or a call function which has an inner attribute object. 
        
        if isinstance(node.value, ast.Attribute): # THis is used for assignement statements
            attr = node.value.attr # here the attribute means that np.attr it self
            attr_name = node.value.value.id
            # print(f'Value: {attr_name}.{attr}')
        elif isinstance(node.value, ast.Call): # These come into play when we affect elements like numpy, e.g. a = np.int32(56)
            if isinstance(node.value.func, ast.Name):
                attr_name = node.value.func.id
            elif isinstance(node.value.func, ast.Attribute):
                attr_name = f'{node.value.func.value.id}.{node.value.func.attr}'
            value = node.value.args[0].value
            # print(f'Value: {attr_name}({value})')

    if isinstance(node, ast.FunctionDef):
        # print(node.lineno, node.col_offset, node.end_lineno, node.end_col_offset)
        # This pratically represents the lines where the function def starts as such:
        # - 20 lineno is the line where the function defintion starts in our case the 'with open' part(1 based index)
        # - 0 col_offset correponds to where the node starts (0 based index) here def declaration_intialization
        # - 22 end_lineo correponds to when the lines stops here when the line stops 
        # - 80 end_col_offset corresponds to when the column stops here at 80 which means at the end of the last line 

        # Retrieve all the children nodes
        func = node.__dict__
        func.pop('type_comment',None)
        
        tmp = {}
        # These lines are helpful when we want to insert elements inside or from the function ending
        tmp['lineno'] = func['lineno']
        tmp['col_offset'] = func['col_offset']
        tmp['end_lineno'] = func['end_lineno']
        tmp['end_col_offset'] = func['end_col_offset']

        # THe function 

In [40]:
def ast_walk(node, node_type):
    """
    Recursively walks an AST tree, yielding all nodes or nodes of a specific type.
    """
    if node_type is None or isinstance(node, node_type):
        yield node
    for child in ast.iter_child_nodes(node):
        yield from ast_walk(child, node_type)

In [41]:
# print(ast.dump(global_python_template,indent=4))
assign_nodes = convert_SPECIFICATION_PART(cls.dec_global,cls_mode=False)
# print(ast.dump(assign_nodes[0],indent=4))

INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: tmc_litt_wet_mea
INFO:root:Successfully removed INTENT and SAVE attributes from statements
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: mcl
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: humrelv
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: tmc
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: frac_bare_ns
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: tmc_litt_dry_mea
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm) :: soilmoist_liquid
INFO:root:Successfully removed INTENT and SAVE attributes from statements
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: soil_wet_litter
INFO:root:Combined statement: INTEGER(KIND = 

In [42]:
def pre_init_variables(code_template):

    pre_init = []
    class_exist = any(ast_walk(code_template,ast.ClassDef))
    if class_exist:
        functions_spec = ast_walk(code_template,ast.FunctionDef)
        for functions in functions_spec:
            if functions.name == "__init__":
                assign_stmt = ast.iter_child_nodes(functions)
                for assign_ in assign_stmt:
                    print(assign_.targets[0].attr)
    else:
        nodes = ast_walk(code_template,ast.Assign)
        for node in nodes:
            pre_init.append(node.targets[0].id)

    return pre_init
init_variables = pre_init_variables(global_python_template)
print(init_variables)

['i_std', 'r_std', 'nsnow', 'nslm', 'nvm', 'nstm', 'kjpindex', 'ier', 'ic0', 'ic', 'icr', 'start_time', 'stop_time']


In [43]:
# Some variable might depend upon others that might require their intialization being done after the intialization of the 
# scalar value
import re
def search_dependant_variables(pre_init_variables) -> Dict:
    subroutine_name = "hydrol_diag_soil"
    dependant_variables = {} # THe variable that is the dependant of the other which has the key as the dependant and the values 
    # the different dependees 
    combined_stmt = None
    for key in cls.dec_global[subroutine_name].keys():
        dependees = []
        declarations = cls.dec_global[subroutine_name][key]
        alloc_spec = any([alloc for alloc in walk(declarations, F23.Attr_Spec) if alloc.string == "ALLOCATABLE"])
        if len(declarations) == 2 and alloc_spec:
            combined_stmt = Processor().combine_allocate_declaration(declarations)
        
        if combined_stmt:
            dimensions_spec_list = walk(walk(combined_stmt,F23.Dimension_Attr_Spec),F23.Explicit_Shape_Spec_List) 
            entity_dec_name = walk(combined_stmt,F23.Entity_Decl)[0].children[0]
            
            # Now we verify if one of these variables has initialization as None
            for arg in dimensions_spec_list[0].children: # This allows to handle cases such as imax:imin type 
                
                limits = arg.tostr().split(' : ')
                lb = limits[0] 
                
                ub = limits[1] if len(limits) > 1 else None
                # print(lb,ub)
                # we now verify that the upper bound/lower bound 's shape is present within the pre init variables and the declarations
                if lb is not None:
                    dec = cls.dec_global[subroutine_name].get(lb,None)
                    if dec is None and lb in pre_init_variables: # This means that the variables is present in the pre init variables
                        continue 
                    elif dec is not None and lb not in pre_init_variables:
                        # we verify the initalization of these shapes
                        for elements in dec:
                            entity_decl_list = walk(elements,F23.Entity_Decl_List)[0]
                            for entity_dec in entity_decl_list.children:
                                _, _,_, initialization = entity_dec.children
                                if initialization is None:
                                    dependees.append(lb)

                if ub is not None:
                    dec = cls.dec_global[subroutine_name].get(ub, None)
                    if dec is None and ub in pre_init_variables: # This means that the variables is present in the pre init variables
                        continue 
                    elif dec is not None and ub not in pre_init_variables:
                        # we verify the initalization of these shapes
                        for elements in dec:
                            entity_decl_list = walk(elements,F23.Entity_Decl_List)[0]
                            for entity_dec in entity_decl_list.children:
                                _, _,_, initialization = entity_dec.children
                                if initialization is None:
                                    dependees.append(ub)
                        
            if len(dependees) > 0:
                dependant_variables[entity_dec_name.string] = dependees

    return dependant_variables
dependant_variables = search_dependant_variables(init_variables)
print(dependant_variables)

INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: tmc_litt_wet_mea
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: mcl
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: humrelv
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: tmc
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: frac_bare_ns
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: tmc_litt_dry_mea
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm) :: soilmoist_liquid
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: soil_wet_litter
INFO:root:Combined statement: INTEGER(KIND = i_std), DIMENSION(kjpindex, nstm) :: mask_soiltile
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot_old
INFO:root:Com

{'k_lin': ['imax']}


In [44]:
class_template = """

import numpy as np
import os
from scipy.io import FortranFile

class global_module:
    def __init__(self):
        i_std = np.int32   
        r_std = np.float64
        
        self.nsnow = i_std(3)
        self.nslm = i_std(11)
        self.nvm = i_std(15)
        self.nstm = i_std(3)
        self.kjpindex = i_std(4717)
        
        self.ier = i_std(0)
        self.ic0 = i_std(0)
        self.ic = i_std(0)
        self.icr = r_std(0.0)
        self.start_time = r_std(0.0)
        self.stop_time = r_std(0.0)

    def declaration_initialization(self):
        pass 
"""
# Adding the variables either based on the given index or based on the position of previous AST assign statements
def insert_at(idx:int, ast_node,python_template,method_name:str=None) -> None:

    class_exists = any(ast_walk(python_template,ast.ClassDef))
    pos = None 
    if class_exists:
            functions_spec = ast_walk(python_template,ast.FunctionDef)
            for functions in functions_spec:
                if not method_name:
                    logging.info(f"Since argument method_name is: {method_name}, defaulting to the __init__ method")
                    if functions.name == "__init__":
                        assign_statement = [pos for pos, assign in enumerate(ast.iter_child_nodes(functions)) if isinstance(assign, ast.Assign)]
                        pos = assign_statement[-1]
                        if idx:
                            if idx < pos:
                                position = pos + 1
                                logging.warning(f'The given index:{idx} is too small that it will be placed before present variables, WILL BE USING previous known ast Assign position')
                                functions.body.insert(position, ast_node)
                            else:
                                functions.body.insert(idx,ast_node)
                        else:
                            position = pos + 1
                            logging.info(f'Since no index is given, WILL BE USING previous known ast Assign position')
                            functions.body.insert(position, ast_node)
                else:
                    logging.info(f"Since argument method_name is: {method_name}, placing the assign statement inside of the method")
                    if functions.name == method_name:
                        assign_statement = [pos for pos, assign in enumerate(ast.iter_child_nodes(functions)) if isinstance(assign, ast.Assign)]
                        pos = assign_statement[-1]
                        if idx:
                            if idx < pos:
                                position = pos + 1
                                logging.warning(f'The given index:{idx} is too small that it will be placed before present variables, WILL BE USING previous known ast Assign position')
                                functions.body.insert(position, ast_node)
                            else:
                                functions.body.insert(idx,ast_node)
                        else:
                            position = pos + 1
                            logging.info(f'Since no index is given, WILL BE USING previous known ast Assign position')
                            functions.body.insert(position, ast_node)
            
                        
    else: 
        if method_name:
            functions = [func for func in ast_walk(python_template,ast.FunctionDef) if func.name == method_name][0]
            logging.info(f"Since argument method_name is: {method_name}, placing the assign statement inside of the method")
            
            assign_statement = [pos for pos, assign in enumerate(ast.iter_child_nodes(functions)) if isinstance(assign, ast.Assign)]
            pos = assign_statement[-1]
            if idx:
                if idx < pos:
                    position = pos + 1
                    logging.warning(f'The given index:{idx} is too small that it will be placed before present variables, WILL BE USING previous known ast Assign position')
                    functions.body.insert(position, ast_node)
                else:
                    functions.body.insert(idx,ast_node)
            else:
                position = pos + 1
                logging.info(f'Since no index is given, WILL BE USING previous known ast Assign position')
                functions.body.insert(position, ast_node)
        else:
            # Retrieve the list of assign statements in which we can ensure that the idx given is suitable to be placed and the 
            # last assign statement as this would be pos from which we append the next set of variables:
            assign_statement = [pos for pos, assign in enumerate(ast.iter_child_nodes(python_template)) if isinstance(assign, ast.Assign)]
            # print(assign_statement)
            pos = assign_statement[-1]
            
            if idx:
                if idx < pos:
                    position = pos + 1
                    logging.warning(f'The given index:{idx} is too small that it will be placed before present variables, WILL BE USING previous known ast Assign position')
                    python_template.body.insert(position, ast_node)
                else:
                    python_template.body.insert(idx,ast_node)
            else:
                position = pos + 1
                logging.info(f'Since no index is given, WILL BE USING previous known ast Assign position')
                python_template.body.insert(position, ast_node)
    
    # What this does it fix the missing location(lineno,end_lineno,col_offset,end_col_offset) based on the parent node
    # https://docs.python.org/3/library/ast.html#ast.fix_missing_locations 
    ast.fix_missing_locations(ast_node)
    

# DO it in two steps first the declared and intializd variables and then the variable order
# The declared and intialized variables 

code_template = ast.parse(global_python_template)

diff = list(set(
    assign.targets[0].id if isinstance(assign.targets[0], ast.Name) else assign.targets[0].attr
    for assign in assign_nodes) - set(variable_order))
name = None

if len(diff) != 0:
    for assign_node in assign_nodes:
        if isinstance(assign_node.targets[0],ast.Name):
            name = assign_node.targets[0].id 
        else:
            name = assign_node.targets[0].attr
        if name in diff:
            insert_at(None,assign_node,global_python_template)

# Now all the declared and not intialized variables
assign_node_names = [assign.targets[0].id if isinstance(assign.targets[0], ast.Name) else assign.targets[0].attr  for assign in assign_nodes]
for var in variable_order:
    for assign_node in assign_nodes:
        if isinstance(assign_node.targets[0],ast.Name):
            name = assign_node.targets[0].id 
        else:
            name = assign_node.targets[0].attr
            
        if var == name and var not in list(dependant_variables.keys()):
            insert_at(None,assign_node,global_python_template)

print(ast.unparse(global_python_template))


INFO:root:Since no index is given, WILL BE USING previous known ast Assign position
INFO:root:Since no index is given, WILL BE USING previous known ast Assign position
INFO:root:Since no index is given, WILL BE USING previous known ast Assign position
INFO:root:Since no index is given, WILL BE USING previous known ast Assign position
INFO:root:Since no index is given, WILL BE USING previous known ast Assign position
INFO:root:Since no index is given, WILL BE USING previous known ast Assign position
INFO:root:Since no index is given, WILL BE USING previous known ast Assign position
INFO:root:Since no index is given, WILL BE USING previous known ast Assign position
INFO:root:Since no index is given, WILL BE USING previous known ast Assign position
INFO:root:Since no index is given, WILL BE USING previous known ast Assign position
INFO:root:Since no index is given, WILL BE USING previous known ast Assign position
INFO:root:Since no index is given, WILL BE USING previous known ast Assign p

import numpy as np
from scipy.io import FortranFile
import os
i_std = np.int32
r_std = np.float64
nsnow = i_std(3)
nslm = i_std(11)
nvm = i_std(15)
nstm = i_std(3)
kjpindex = i_std(4717)
ier = i_std(0)
ic0 = i_std(0)
ic = i_std(0)
icr = r_std(0.0)
start_time = r_std(0.0)
stop_time = r_std(0.0)
un = np.float64(1.0)
imin = np.int32(1)
zero = np.float64(0.0)
iice = np.int32(1)
min_sechiba = np.float64(1e-08)
huit = np.float64(8.0)
nnobio = np.int32(1)
trois = np.float64(3.0)
imax = np.int32(0)
ok_freeze_cwrr = np.bool(False)
ae_ns = np.empty((kjpindex, nstm), dtype=np.float64)
dh = np.empty((nslm,), dtype=np.float64)
dr_ns = np.empty((kjpindex, nstm), dtype=np.float64)
dz = np.empty((nslm,), dtype=np.float64)
frac_bare_ns = np.empty((kjpindex, nstm), dtype=np.float64)
humrelv = np.empty((kjpindex, nvm, nstm), dtype=np.float64)
humtot = np.empty((kjpindex,), dtype=np.float64)
mask_soiltile = np.empty((kjpindex, nstm), dtype=np.int32)
mc = np.empty((kjpindex, nslm, nstm), dtype=np.float64)


In [45]:
# Separate between the dec_global values among those who are going to intialized but are declared, the scalar/logical variables(immutable) and the tables which
# are set as numpy arrays(mutable objects) for these tables we will directly retrieve their dimensions 
def separate_scalar(variable_order):
    scalar = []
    for var in variable_order:
        dec_statement = cls.dec_global[subroutine_key][var]

        init_spec = any(walk(dec_statement, F23.Initialization))
        alloc_spec = any([alloc for alloc in walk(dec_statement, F23.Attr_Spec) if alloc.string == "ALLOCATABLE"])
        if not init_spec and not alloc_spec:
            scalar.append(var)
    return scalar

def retrieve_table_info(variable_order):
    tables = {}
    args_ = None
    combined_stmt = None
    logging.info(f'Combining allocate statement together to retrieve dimension shape inside the retrieve_table_info method')
    for var in variable_order:
        dec_statement = cls.dec_global[subroutine_key][var]
        alloc_spec = any([alloc for alloc in walk(dec_statement, F23.Attr_Spec) if alloc.string == "ALLOCATABLE"])
        if len(dec_statement) == 2 and alloc_spec:
            combined_stmt = Processor().combine_allocate_declaration(dec_statement)
        
            if combined_stmt:
                dimensions_spec_list = walk(walk(combined_stmt,F23.Dimension_Attr_Spec),F23.Explicit_Shape_Spec_List) 
                children = dimensions_spec_list[0].children
                args_ = [var.string for var in children if var is not None]
    
            tables[var] = args_
    return tables
        
scalar = separate_scalar(variable_order)

In [46]:
def prepare_read_code_template(assign_nodes, variable_order):
    read_code_template = f"""
path = f'/data/ssivanes/Fgpt/benchmark/{subroutine_key}/global.bin'
ffile = FortranFile(path, 'r')

    """
    read_ast = python_parser(read_code_template)
    # print(ast.dump(read_ast,indent=4))
    var = None
    var_name = None
    target = None
    for var in variable_order:
        for assign_node in assign_nodes:
            if isinstance(assign_node.targets[0], ast.Name):
                var_name = assign_node.targets[0].id
                target = ast.Name(id=var_name,ctx = ast.Store())
            else:
                var_name = assign_node.targets[0].attr
                target = ast.Attribute(
                    value = ast.Name(id='self',ctx=ast.Load()),
                    attr = var_name,
                    ctx = ast.Load()
                )
            if var_name == var:
                if len(assign_node.value.keywords) == 0: # This means they are just intergers,reals or logical values mostly scalars 
                    attr_type = assign_node.value.func.attr
                    if attr_type == "int32" or attr_type == "bool":
                        # var = ast.parse(f"{assign_node.targets[0].id} = ffile.read_ints(np.int32)[0] ")
                        subscript_format = ast.Subscript(
                                                value = ast.Call(
                                                    func = ast.Attribute(
                                                        value= ast.Name(id = 'ffile',ctx=ast.Load()),
                                                        attr='read_ints',
                                                        ctx = ast.Load()),
                                                    args = [
                                                        ast.Attribute(
                                                            value = ast.Name(id = 'np',ctx=ast.Load()),
                                                            attr = "int32",
                                                            ctx=ast.Load()) ],
                                                    keywords = []),
                                                slice = ast.Constant(value=0),
                                                ctx=ast.Load()
                                                
                                            )
                        var = ast.Assign(
                            targets = [target],
                            value =  subscript_format
                        )
                        
                
                else: # This means that they all are arrays
                    attr_type = assign_node.value.keywords[0].value.attr
                    # print(arr_shape)
                    call_stmt = None
                    if attr_type == "float64":
                        # print(assign_node.targets[0].id)
                        # var = ast.parse(f'{assign_node.targets[0].id}[:] = ffile.read_record(np.float64).reshape((),order="F")')
                        call_stmt = ast.Call(
                            func= ast.Attribute(
                                value= ast.Call(
                                    func=ast.Attribute(
                                        value=ast.Name(id='ffile', ctx=ast.Load()),
                                        attr='read_record',
                                        ctx=ast.Load()),
                                    args=[
                                        ast.Attribute(
                                            value=ast.Name(id='np', ctx=ast.Load()),
                                            attr=attr_type,
                                            ctx=ast.Load())],
                                    keywords=[]),
                                attr='reshape',
                                ctx=ast.Load()),
                            args=[assign_node.value.args[0]],
                            keywords=[ ast.keyword( arg='order', value=ast.Constant(value='F'))])
                         
                    elif attr_type == "int32":
                        # var = ast.parse(f'{assign_node.targets[0].id}[:] = ffile.read_record(np.int32).reshape((),order="F")')
                        call_stmt = ast.Call(
                            func= ast.Attribute(
                                value= ast.Call(
                                    func=ast.Attribute(
                                        value=ast.Name(id='ffile', ctx=ast.Load()),
                                        attr='read_record',
                                        ctx=ast.Load()),
                                    args=[
                                        ast.Attribute(
                                            value=ast.Name(id='np', ctx=ast.Load()),
                                            attr=attr_type,
                                            ctx=ast.Load())],
                                    keywords=[]),
                                attr='reshape',
                                ctx=ast.Load()),
                            args=[assign_node.value.args[0]],
                            keywords=[ ast.keyword( arg='order', value=ast.Constant(value='F'))])
                        
                    var = ast.Assign(
                        targets=[
                        ast.Subscript(
                            value=target,
                            slice=ast.Slice(),
                            ctx=ast.Store())],
                        value = call_stmt
                        
                    )
                
                read_ast.body.append(var)
    ast.fix_missing_locations(read_ast)
    return read_ast
# prepare_read_code_template(assign_nodes,variable_order)
read_ast = prepare_read_code_template(assign_nodes,variable_order)
# print(ast.dump(read_ast,indent=4))
print(ast.unparse(read_ast))

Code valid
path = f'/data/ssivanes/Fgpt/benchmark/hydrol_diag_soil/global.bin'
ffile = FortranFile(path, 'r')
imax = ffile.read_ints(np.int32)[0]
ok_freeze_cwrr = ffile.read_ints(np.int32)[0]
ae_ns[:] = ffile.read_record(np.float64).reshape((kjpindex, nstm), order='F')
dh[:] = ffile.read_record(np.float64).reshape((nslm,), order='F')
dr_ns[:] = ffile.read_record(np.float64).reshape((kjpindex, nstm), order='F')
dz[:] = ffile.read_record(np.float64).reshape((nslm,), order='F')
frac_bare_ns[:] = ffile.read_record(np.float64).reshape((kjpindex, nstm), order='F')
humrelv[:] = ffile.read_record(np.float64).reshape((kjpindex, nvm, nstm), order='F')
humtot[:] = ffile.read_record(np.float64).reshape((kjpindex,), order='F')
k_lin[:] = ffile.read_record(np.float64).reshape(( imax - imin  + 1, nslm, kjpindex), order='F')
mask_soiltile[:] = ffile.read_record(np.int32).reshape((kjpindex, nstm), order='F')
mc[:] = ffile.read_record(np.float64).reshape((kjpindex, nslm, nstm), order='F')
mcl[:] = ffile

In [47]:
def init_dependant_variables(dependant_variables,read_ast,assign_nodes): 
    # Method use to init variables after the dependecies have initalized
    
    # Get all assign statements
    assign_stmts = ast_walk(read_ast,ast.Assign)
    var_name = None
    for key in list(dependant_variables.keys()):
        pos = 0 # This will allows us to find after which variable should we place the init of the dependant variables
        # Since each depandant variables might have multiple dependee variables which are init at different locations as such we try to find the
        # furthest/last positin of the variable dependee 
        dependees = dependant_variables[key]
        for i, stmt in enumerate(assign_stmts):
            if isinstance(stmt.targets[0], ast.Name):
                # print(stmt.lineno)
                if stmt.targets[0].id in dependees:
                    if i > pos: # THis way we retrieve the maximum dependee position
                        pos = i
            elif isinstance(stmt.targets[0], ast.Attribute):
                if stmt.targets[0].attr in dependees:
                    if i > pos:
                        pos = i
                        
        for assign_node in assign_nodes:
            if isinstance(assign_node.targets[0], ast.Name):
                var_name = assign_node.targets[0].id
            elif isinstance(assign_node.targets[0], ast.Attribute):
                    var_name = assign_node.targets[0].attr
            if var_name == key:
                read_ast.body.insert(pos+1,assign_node)

    ast.fix_missing_locations(read_ast)
    return read_ast
# read_ast = init_dependant_variables(dependant_variables,read_ast,assign_nodes)
# print(ast.unparse(read_ast))

In [48]:
def convert_global_read_subroutine(variable_order,assign_nodes,dependant_variables) -> None:
    # The variable order will be retrieved since we the instance of the isolator class which use the processor and extractor class
    function_def = ast_walk(global_python_template,ast.FunctionDef)
    # print(ast.dump(read_ast,indent=4))
    
    # Retrieved the scalar and table information 
    scalar = separate_scalar(variable_order)

    read_ast = prepare_read_code_template(assign_nodes, variable_order)
    read_ast = init_dependant_variables(dependant_variables,read_ast,assign_nodes)
    for functions in function_def:
        if functions.name == "declaration_initialization": # IF we find the declaration intiailization method to read and fill tables
            if scalar:
                tree = ast.parse(f"global {', '.join(scalar + list(dependant_variables.keys()))}")
                
                functions.body.append(tree.body[0])
                
            # functions.body.append(f'global {scalar[:len(scalar)]}')
            for elem in ast.iter_child_nodes(read_ast):
                 functions.body.append(elem)
        else:
            print(functions.name)
    
    ast.fix_missing_locations(global_python_template)
convert_global_read_subroutine(variable_order,assign_nodes,dependant_variables)

Code valid


In [49]:
# print(ast.dump(global_python_template,indent=4))
print(ast.unparse(global_python_template))

import numpy as np
from scipy.io import FortranFile
import os
i_std = np.int32
r_std = np.float64
nsnow = i_std(3)
nslm = i_std(11)
nvm = i_std(15)
nstm = i_std(3)
kjpindex = i_std(4717)
ier = i_std(0)
ic0 = i_std(0)
ic = i_std(0)
icr = r_std(0.0)
start_time = r_std(0.0)
stop_time = r_std(0.0)
un = np.float64(1.0)
imin = np.int32(1)
zero = np.float64(0.0)
iice = np.int32(1)
min_sechiba = np.float64(1e-08)
huit = np.float64(8.0)
nnobio = np.int32(1)
trois = np.float64(3.0)
imax = np.int32(0)
ok_freeze_cwrr = np.bool(False)
ae_ns = np.empty((kjpindex, nstm), dtype=np.float64)
dh = np.empty((nslm,), dtype=np.float64)
dr_ns = np.empty((kjpindex, nstm), dtype=np.float64)
dz = np.empty((nslm,), dtype=np.float64)
frac_bare_ns = np.empty((kjpindex, nstm), dtype=np.float64)
humrelv = np.empty((kjpindex, nvm, nstm), dtype=np.float64)
humtot = np.empty((kjpindex,), dtype=np.float64)
mask_soiltile = np.empty((kjpindex, nstm), dtype=np.int32)
mc = np.empty((kjpindex, nslm, nstm), dtype=np.float64)


In [51]:
from scipy.io import FortranFile
import numpy as np
path = f'benchmark/{subroutine_key}/global.bin'

ffile = FortranFile(path, 'r')
kjpindex = 4717
nstm = 3
nslm = 11
nvm = 15
nsnow = 3
imax = 51
imin = 1

imax = ffile.read_ints(np.int32)[0]
ok_freeze_cwrr = ffile.read_ints(np.int32)[0]
ae_ns = ffile.read_record(np.float64).reshape((kjpindex,nstm),order='F')
dh = ffile.read_record(np.float64).reshape((nslm,),order="F")
dr_ns = ffile.read_record(np.float64).reshape((kjpindex,nstm),order="F")
dz = ffile.read_record(np.float64).reshape((nslm,),order="F")
frac_bare_ns = ffile.read_record(np.float64).reshape((kjpindex,nstm),order="F")
humrelv = ffile.read_record(np.float64).reshape((kjpindex,nvm,nstm),order="F")
humtot = ffile.read_record(np.float64).reshape((kjpindex,),order="F")
k_lin = ffile.read_record(np.float64).reshape((imax - imin + 1, nslm,kjpindex),order="F")

# print(humtot.shape)
# print(k_lin)

In [52]:
k_lin = np.empty(( imax - imin +1, nslm, kjpindex), dtype=np.float64)
imax = np.int32(0)

def declaration_initialization():
    print('--- add the declaration and initialization in module global ---')
    global imax, ok_freeze_cwrr, k_lin

    for var in [imax,k_lin]:
        if var.dtype == np.float64 and isinstance(var, np.ndarray):
            print(var)

declaration_initialization()

--- add the declaration and initialization in module global ---
[[[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 ...

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]


In [53]:
#Transform from AST Fortran to AST python class approach. 
def transform_to_class(assign_nodes,variable_order,dependant_variables):
    class_template = """

import numpy as np
import os
from scipy.io import FortranFile

class global_module:
    def __init__(self):
        i_std = np.int32   
        r_std = np.float64
        
        self.nsnow = i_std(3)
        self.nslm = i_std(11)
        self.nvm = i_std(15)
        self.nstm = i_std(3)
        self.kjpindex = i_std(4717)
        
        self.ier = i_std(0)
        self.ic0 = i_std(0)
        self.ic = i_std(0)
        self.icr = r_std(0.0)
        self.start_time = r_std(0.0)
        self.stop_time = r_std(0.0)

    def declaration_initialization(self):
        pass 
"""
    # Instead of using global_python_template use instead teh methods within the function itself. 
    
    class_tree = ast.parse(class_template)
    functions_spec = ast_walk(class_tree,ast.FunctionDef)

    assign_stmt = None
    for functions in functions_spec:
        if functions.name == "__init__":
            # DO it in two steps first the declared and intializd variables and then the variable order
            # The declared and intialized variables 
            diff = list(set([assign.targets[0].attr for assign in assign_nodes]) - set(variable_order))
            if len(diff) != 0:
                for assign_node in assign_nodes:
                    if assign_node.targets[0].attr in diff:
                        insert_at(None,assign_node,class_tree,"__init__")
                        # functions.body.append(assign_stmt)
                
            # Now all the declared and not intialized variables
            
            assign_node_names = [assign.targets[0].attr for assign in assign_nodes]
            for var in variable_order:
                for assign_node in assign_nodes:
                    if var == assign_node.targets[0].attr and var not in list(dependant_variables.keys()):
                       insert_at(None,assign_node,class_tree,"__init__")
            
        elif functions.name == "declaration_initialization":
            if isinstance(functions.body[0], ast.Pass):
                functions.body.pop(0)
            
            read_ast = prepare_read_code_template(assign_nodes, variable_order)
            read_ast = init_dependant_variables(dependant_variables,read_ast,assign_nodes)

            for elem in ast.iter_child_nodes(read_ast):
                 functions.body.append(elem)
            
    

    ast.fix_missing_locations(class_tree)
    
    print(ast.unparse(class_tree))


# transform_to_class(assign_nodes,variable_order,dependant_variables)